# PosterEval Walkthrough

Interactive demo of the offline evaluation pipeline for streaming artwork personalization.

This notebook estimates how a candidate poster policy would have performed using only logs from the current contextual bandit system, then checks that estimate against a randomized holdout. Matching the holdout is a check that the click-rate math is working on this simulation, not a claim about live user traffic.

It uses 8,000 impressions so the cells finish quickly. The README and the Airflow pipeline use the 20,000-impression demo; those published numbers will not match this notebook.

This notebook:

1. Simulates contextual-bandit impressions (epsilon-greedy mixture propensity).
2. Fits an XGBoost model for the chance the old policy shows a poster.
3. Estimates the candidate policy's click rate with IPS, SNIPS, and Doubly Robust.
4. Checks those estimates against a randomized holdout.

Run from the `poster-eval` directory (or this `notebooks/` folder).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd()
root = cwd if (cwd / "src").exists() else cwd.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.ope.estimators import run_all_estimators
from src.ope.evaluate import compute_candidate_propensities
from src.propensity.train import fit_reward_model, train_propensity_model
from src.simulator.bandit import PolicyWeights
from src.simulator.generate_logs import generate_bandit_logs
from src.validation.holdout import validate_policy

SEED = 42
N_IMPRESSIONS = 8000  # smaller than the README / Airflow 20k demo
print("project root:", root)

## 1. Simulate Logged Impressions

Each row is one poster impression:

- **context:** user genre affinity, device, title popularity tier
- **poster shown:** which of $K$ artwork variants was displayed (`artwork_id`, `variant_index`)
- **propensity:** chance the current, baseline policy would show that poster, using the epsilon-greedy mixture $\varepsilon/K + (1-\varepsilon)\pi_{\mathrm{softmax}}$
- **click:** no click / click (0 or 1); stored in the `reward` column
- **`is_randomized`:** explore bucket used later as holdout ground-truth data

In [ ]:
logs = generate_bandit_logs(
    n_impressions=N_IMPRESSIONS,
    n_titles=40,
    n_users=80,
    seed=SEED,
)

preview_cols = [
    "impression_id",
    "artwork_id",
    "variant_index",
    "logging_propensity",
    "reward",
    "is_randomized",
    "popularity_tier",
]
print(logs[preview_cols].head().to_string(index=False))
print(
    f"{len(logs):,} impressions | "
    f"mean click rate={logs['reward'].mean():.3f} | "
    f"randomized={logs['is_randomized'].mean():.1%} | "
    f"min propensity={logs['logging_propensity'].min():.4f}"
)


## 2. Train the Propensity Model

Production logs often lack propensity values, so we show how they can be estimated by fitting an XGBoost model on context features and then employing softmax over the $K$ artwork variants.

On this simulation the logger is not far from uniform ($K = 5$), so the model's log-loss is usually only a hair better than the uniform baseline ($-\log(1/K)$). That small edge is expected; it is not a headline win. Coverage above the $10^{-3}$ clip floor keeps IPS from exploding.

In [ ]:
propensity = train_propensity_model(logs, model_type="xgboost", seed=SEED)
metrics = propensity.metrics
print("model:", propensity.model_type, "| classes:", propensity.n_classes)
for key in ("log_loss", "uniform_log_loss", "coverage_above_floor", "mae_vs_oracle", "calibration_mae"):
    if key in metrics:
        print(f"  {key}: {metrics[key]:.6f}")

edge = metrics["uniform_log_loss"] - metrics["log_loss"]
print(f"log-loss edge vs uniform: {edge:.6f} (small on this sim; logger is close to uniform)")


## 3. Off-Policy Estimators

A new candidate policy (`bandit_v2`, stronger genre matching than the baseline) is compared to the current logging policy. Off-policy evaluation reweights the logs by the ratio of how likely the new policy would have been to show that poster versus how likely the old policy was to show it.

The output table below the code summarizes the full 8,000-row sample. The next section repeats IPS/SNIPS/DR on non-holdout rows only, which is the README-style holdout check.

| Estimator | Idea |
|-----------|------|
| **IPS** | click × (new / old) — unbiased, high variance |
| **SNIPS** | Self-normalized IPS |
| **DR** | IPS residual plus a click model on the poster that was actually shown (not a sum over every variant) |

In [ ]:
candidate_p = compute_candidate_propensities(logs, PolicyWeights.candidate_v2(), seed=SEED)
mu = fit_reward_model(logs, seed=SEED)
oracle_p = logs["logging_propensity"].to_numpy()

candidate_results = run_all_estimators(logs, candidate_p, reward_predictions=mu)
logging_results = run_all_estimators(logs, oracle_p, reward_predictions=mu)

def _by_name(results):
    return {r.estimator: r.value for r in results}

cand = _by_name(candidate_results)
logg = _by_name(logging_results)
lift = cand["SNIPS"] - logg["SNIPS"]

summary = pd.DataFrame(
    {
        "logging": logg,
        "bandit_v2": cand,
        "lift": {k: cand[k] - logg[k] for k in cand},
    }
)
print(summary.round(4).to_string())
print(f"SNIPS lift vs logging (all {len(logs):,} rows): {lift:+.4f}")


## 4. Holdout Validation

On `is_randomized=True` rows, a poster was shown uniformly at $\pi_e = 1/K$. Holdout IPS on that slice is the ground-truth click rate for the candidate policy. Compare it to IPS, SNIPS, and DR on the non-holdout rows (those still use the baseline logging propensity).

In [ ]:
validation = validate_policy(logs, candidate_p, reward_predictions=mu)
holdout_value = validation["holdout_ground_truth"]["value"]

compare = pd.DataFrame(validation["estimators_vs_holdout"])
keep = [c for c in ("estimator", "value", "ground_truth", "absolute_bias", "signed_error") if c in compare.columns]
print(compare[keep].round(4).to_string(index=False))

print(
    f"holdout IPS={holdout_value:.4f} | "
    f"SNIPS |bias|={validation['snips_absolute_bias']:.4f} | "
    f"RMSE={validation['rmse_across_estimators']:.4f} | "
    f"n_holdout={validation['n_holdout']}"
)

## 5. Click Rate vs Holdout

Left: estimated click rate on the same non-holdout slice as the table above (logging SNIPS, candidate IPS / SNIPS / DR) next to holdout IPS.

Right: candidate SNIPS versus holdout IPS by catalog tier (`head` / `mid` / `tail`). Artwork personalization is often expected to matter more on unfamiliar tail titles; on a small simulated run the gap can show up anywhere, including the head.

In [ ]:
tier_snips = {}
for tier in ("head", "mid", "tail"):
    block = validation["stratified_by_tier"].get(tier) or {}
    rows = block.get("estimators") or []
    snips_row = next((r for r in rows if r.get("estimator") == "SNIPS"), None)
    if snips_row:
        tier_snips[tier] = {
            "candidate_snips": snips_row["value"],
            "holdout": snips_row.get("ground_truth", holdout_value),
        }

exploit_mask = ~logs["is_randomized"].to_numpy()
logging_on_exploit = {
    r.estimator: r.value
    for r in run_all_estimators(
        logs.loc[exploit_mask],
        logs.loc[exploit_mask, "logging_propensity"].to_numpy(),
        reward_predictions=mu[exploit_mask],
    )
}
candidate_on_exploit = {
    row["estimator"]: row["value"] for row in validation["estimators_vs_holdout"]
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

labels = ["logging\nSNIPS", "IPS", "SNIPS", "DR", "holdout\nIPS"]
values = [
    logging_on_exploit["SNIPS"],
    candidate_on_exploit["IPS"],
    candidate_on_exploit["SNIPS"],
    candidate_on_exploit["DR"],
    holdout_value,
]
colors = ["#7a7a7a", "#4c78a8", "#f58518", "#54a24b", "#e45756"]
axes[0].bar(labels, values, color=colors)
axes[0].set_ylabel("estimated click rate")
axes[0].set_title("Non-holdout estimates vs holdout IPS")
axes[0].axhline(holdout_value, color="#e45756", linestyle="--", linewidth=1, alpha=0.8)

if tier_snips:
    tiers = list(tier_snips)
    x = np.arange(len(tiers))
    w = 0.35
    axes[1].bar(x - w / 2, [tier_snips[t]["candidate_snips"] for t in tiers], w, label="candidate SNIPS", color="#f58518")
    axes[1].bar(x + w / 2, [tier_snips[t]["holdout"] for t in tiers], w, label="holdout IPS", color="#e45756")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(tiers)
    axes[1].set_title("Candidate SNIPS vs holdout IPS by tier")
    axes[1].legend()
else:
    axes[1].set_title("Not enough per-tier holdout rows — bump n_impressions")

fig.tight_layout()
plt.show()
